# Did CEM see a more direct plan?

Use a GPU runtime. This audit reproduces the original first CEM search for the green upper-goal detour, **b1_g04_s01**, from the five-macro square-band run.

It records all sampled sequences and the returned elite mean. We inspect predicted detour versus terminal MSE, then replay a small group of promising candidates and a scripted direct control from the same start. We record both first arrival inside the success circle and the endpoint after all 25 actions. The original run is read-only.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import io
import json
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/experience_temporal_half_square_v1')
RUN_ID='20261107T150425413490Z'  # The run used in the attached audit.
CASE_ID='b1_g04_s01'
SUCCESS_RADIUS=4.5

## Read the archived commands and observations

The archived macro-actions must exactly match the executed primitive prefix. The diagnostic requires a rollout produced by one planner call.

In [ ]:
def read_json(path):
    return json.loads(path.read_text())

run=RUNS/RUN_ID
stage=read_json(run/'stages/band_1.json')
assert stage['status']=='complete' and stage['condition']['collector']=='raw'
manifest=read_json(run/'experiment.json')['manifest']
cases=manifest['stages'][0]
assert stage['cases_digest']==hashlib.sha256(json.dumps(cases,sort_keys=True,allow_nan=False).encode()).hexdigest()
case=next(c for c in cases if c['case_id']==CASE_ID)
record=read_json(run/f'episodes/band_1/{CASE_ID}.json')
assert record['case']==case and record['condition']==stage['condition'] and record['runtime_id']==stage['runtime_id']
assert record['mpc_updates']==1, 'Select a band 1 episode with exactly one planner call.'
raw_path=(run/record['raw_path']).resolve()
raw_path.relative_to(run.resolve())
data=raw_path.read_bytes()
assert hashlib.sha256(data).hexdigest()==record['raw_sha256'], 'Raw rollout checksum differs.'
raw=torch.load(io.BytesIO(data),map_location='cpu',weights_only=True)
del data
assert raw['case']==case and raw['runtime_id']==stage['runtime_id'] and raw['success']==record['success']
steps=record['micro_steps']
actual_xy=raw['states'][:,:2].numpy()
assert len(actual_xy)==len(raw['frames'])==steps+1
assert np.array_equal(raw['primitive_steps'].numpy(),np.arange(steps+1))
assert np.array_equal(raw['primitive_actions'].numpy(),np.asarray(record['primitive_actions'],dtype=np.float32))
planning=manifest['planning']
print('Case:',CASE_ID,'· start:',case['start'],'· goal:',case['goal'])
print('Recorded simulator actions:',steps,'· macro durations:',record['macro_durations'])

## Restore the original models and planning settings

Both model files are verified against the collection runtime, and the original goal tensor is reused. For an exact CEM reproduction, use the same GPU model and PyTorch build shown below. The candidate audit checks that CEM returns the same archived sequence before interpreting its samples.

In [ ]:
import re
import shutil
import subprocess
import sys
from types import SimpleNamespace

assert torch.cuda.is_available(), 'Select a GPU runtime.'
runtime=read_json(run/'runtime.json')
signature=runtime['signature']
stage=read_json(run/'stages/band_1.json')
assert runtime['runtime_id']==stage['runtime_id'], 'Band 1 runtime changed.'

def checkout(url,path,commit,branch=None):
    assert re.fullmatch(r'[a-f0-9]{40}',commit), 'Invalid recorded commit.'
    if not path.exists():
        args=['git','clone']
        if branch:
            args+=['--branch',branch,'--single-branch']
        subprocess.run([*args,url,str(path)],check=True)
    assert subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True).strip()==url
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
    exists=subprocess.run(['git','-C',str(path),'cat-file','-e',commit],capture_output=True)
    if exists.returncode:
        subprocess.run(['git','-C',str(path),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(path),'checkout','--detach',commit],check=True)

REPO=Path('/content/i-jepa-wm-prediction-replay')
DINO=Path('/content/dino_wm')
checkout('https://github.com/kmylonas/i-jepa-wm.git',REPO,runtime['source_commit'],'curriculum-cost-adapter')
checkout('https://github.com/gaoyuezhou/dino_wm.git',DINO,signature['dino_commit'])
subprocess.run([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml',
                'scipy','einops','pillow','imageio','imageio-ffmpeg'],check=True)
sys.path.insert(0,str(REPO))

from src.mylonas_ijepa_wm import plan_pointmaze as visual, plan_pointmaze_proprio as planner, plan_proprio
from src.mylonas_ijepa_wm.curriculum_runtime import PilotRuntime,file_sha256,source_digest
from src.mylonas_ijepa_wm.experience_curriculum import _raw_cost
from src.mylonas_ijepa_wm.environment_adapters import WallAdapter
from src.mylonas_ijepa_wm.environment_configs import EnvironmentConfig
from src.mylonas_ijepa_wm.plan_pointmaze import load_frozen_ijepa_encoder

assert source_digest(REPO/'src')==signature['source'], 'Encoder source differs from collection.'
assert source_digest(DINO/'env/wall')==signature['dino_source'], 'Renderer source differs from collection.'
goal_path=run/'collection_goals.pt'
assert file_sha256(goal_path)==stage['condition']['goal_sha256'], 'Saved collector goal tokens changed.'
targets=torch.load(goal_path,map_location='cpu',weights_only=True)
assert targets['runtime_id']==runtime['runtime_id'], 'Goal tensors belong to another runtime.'
target_tokens=targets['tokens']

device=torch.device('cuda')
variant=signature['runtime_variant']
torch.set_float32_matmul_precision(variant['matmul_precision'])
print('Collection GPU:',variant['device_name'],'· Torch:',signature['torch_version'])
print('Audit GPU:',torch.cuda.get_device_name(device),'· Torch:',str(torch.__version__))

def local_checkpoint(name,source,expected):
    path=Path('/content')/name
    if not path.is_file() or file_sha256(path)!=expected:
        shutil.copy2(source,path)
    assert file_sha256(path)==expected, f'Checksum differs for {source}'
    return path

drive_root=Path('/content/drive/MyDrive')
encoder_path=local_checkpoint('raw-mse-audit-encoder.pth.tar',
    drive_root/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar',signature['models']['encoder'])
wm_path=local_checkpoint('raw-mse-audit-world-model.pt',
    drive_root/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt',signature['models']['world_model'])
encoder=load_frozen_ijepa_encoder(encoder_path,device)
loader=SimpleNamespace(device=device,args=SimpleNamespace(world_model_checkpoint=wm_path),model=None)
model=PilotRuntime.load_world_model(loader).eval().requires_grad_(False)
metadata=loader.metadata
config=EnvironmentConfig(env='wall',wall_layouts=[case['layout']],dino_wm_root=str(DINO))
env=WallAdapter(config,planning['max_macro_steps']*config.actions_per_model_step)
precision=signature['precision']
target=target_tokens[case['goal_index']].to(device).float()
score=_raw_cost(target)

## Reproduce the search and save its candidates

The observer records candidate commands and scores without changing the search. Elite membership is computed with the original selection rule on the same device. The returned elite mean is recorded separately from the sampled candidates.

In [ ]:
def archived_plan(record,raw,stride,action_dim,horizon):
    macro=np.asarray(record['macro_actions'],dtype=np.float32)
    durations=record['macro_durations']
    primitive=raw['primitive_actions'].numpy()
    if (macro.shape!=(horizon,stride*action_dim) or len(durations)!=horizon
            or durations[:-1]!=[stride]*(horizon-1) or not 1<=durations[-1]<=stride
            or sum(durations)!=len(primitive) or not np.isfinite(macro).all()
            or not np.array_equal(macro.reshape(-1,action_dim)[:len(primitive)],primitive)):
        raise ValueError('The full commanded sequence is unavailable or its executed prefix differs.')
    return macro

def reset_case():
    return env.reset(case['seed'],layout=case['layout'],start=case['start'],goal=case['goal'])

stride=config.actions_per_model_step
horizon=planning['planning_horizon']
macro=archived_plan(record,raw,stride,config.micro_action_dim,horizon)
commands=macro.reshape(-1,config.micro_action_dim)
assert np.array_equal(reset_case().frame,raw['frames'][0].numpy()), 'Reset image differs.'
assert np.allclose(env.state()[:2],actual_xy[0],rtol=0,atol=1e-5), 'Reset position differs.'


from unittest.mock import patch

rounds=[]
def observe_candidates(index, actions, costs):
    elites=torch.topk(costs,k=planning['num_elites'],largest=False).indices
    rounds.append((index,actions.cpu(),costs.cpu(),elites.cpu()))

class FirstPlan(Exception):
    pass

returned=[]
def stop_after_planning(step,plan,*_):
    assert step==0
    returned.append(plan)
    raise FirstPlan

original_plan_action=planner.plan_action
def observe_plan_action(*args,**kwargs):
    return original_plan_action(*args,**kwargs,candidate_observer=observe_candidates)

assert plan_proprio.planner is planner
initial=reset_case()
with patch.object(planner,'plan_action',observe_plan_action):
    try:
        plan_proprio.run_episode(env,config,encoder,model,metadata,SimpleNamespace(**planning,cost='latent'),
            device,precision,case['seed'],None,initial=initial,latent_cost_fn=score,plan_observer=stop_after_planning)
    except FirstPlan:
        pass

plan=returned[0]
command_error=float(np.abs(plan.action_sequence.cpu().numpy()-macro).max())
print('Maximum command error versus archived plan:',command_error)
print('Returned predicted terminal MSE:',plan.cost)
assert command_error==0 and np.isclose(plan.cost,record['final_planner_cost'],rtol=0,atol=1e-7), (
    'CEM did not reproduce the archived result. Use the original GPU type and Torch build.')
assert [item[0] for item in rounds]==list(range(planning['cem_iterations']))
assert all(len(actions)==planning['population'] for _,actions,_,_ in rounds)
print('Sampled sequences:',sum(len(actions) for _,actions,_,_ in rounds))

## Compare predicted detour and terminal cost

Predicted geometry is available at macro boundaries. **Outward excursion** is the largest increase above the initial goal distance before first predicted success, or through the horizon if it never predicts success. **Extra path** subtracts the straight displacement from the length of the predicted path to that point. The default low-detour group requires a predicted success, outward excursion at most 1 unit, and extra path at most 2 units. These thresholds are audit settings.

In [ ]:
MAX_OUTWARD=1.
MAX_EXTRA=2.
goal=np.asarray(case['goal'])
initial_latent=visual.encode_frame(encoder,raw['frames'][0].numpy(),device,precision)
mean=metadata['state_mean'].to(device=device,dtype=torch.float32)
std=metadata['state_std'].to(device=device,dtype=torch.float32)
normalized=(raw['states'][0].to(device=device,dtype=torch.float32)-mean)/std
latent_history=initial_latent[None,None].repeat(1,model.num_hist,1,1)
state_history=normalized[None,None].repeat(1,model.num_hist,1)
action_history=torch.zeros(model.num_hist-1,config.macro_action_dim,device=device)

def predict(actions):
    latents,states=planner.rollout_action_sequences(model,latent_history,state_history,action_history,
        actions,device=device,chunk_size=planning['candidate_batch_size'],
        action_mean=metadata.get('action_mean'),action_std=metadata.get('action_std'),precision=precision)
    xy=(states.float()*std+mean).cpu().numpy()[:,:,:2]
    start=np.broadcast_to(actual_xy[0],(len(xy),1,2))
    return np.concatenate([start,xy],axis=1),score(latents[:,-1]).cpu().numpy()

def geometry(xy,first_hit):
    observed=xy if first_hit is None else xy[:first_hit+1]
    distance=np.linalg.norm(observed-goal,axis=1)
    path_length=float(np.linalg.norm(np.diff(observed,axis=0),axis=1).sum())
    return dict(outward=float(distance.max()-distance[0]),path_length=path_length,
                extra=max(0.,path_length-float(np.linalg.norm(observed[-1]-observed[0]))))

def predicted_row(ident,xy,cost,**fields):
    hits=np.flatnonzero(np.linalg.norm(xy-goal,axis=1)<SUCCESS_RADIUS)
    hit=int(hits[0]) if len(hits) else None
    geom=geometry(xy,hit)
    return dict(id=ident,predicted_mse=float(cost),predicted_first_hit=None if hit is None else hit*stride,
                predicted_outward=geom['outward'],predicted_extra=geom['extra'],
                low_detour=hit is not None and geom['outward']<=MAX_OUTWARD and geom['extra']<=MAX_EXTRA,
                **fields)

predicted_rows=[]
commands_by_id={}
state_paths=[]
command_blocks=[]
score_error=0.
batch=planning['candidate_batch_size']
for index,actions,costs,elite_ids in rounds:
    elite=set(elite_ids.tolist())
    command_blocks.append(actions.numpy())
    for start in range(0,len(actions),batch):
        xy,check=predict(actions[start:start+batch])
        state_paths.extend(xy)
        original=costs[start:start+batch].numpy()
        score_error=max(score_error,float(np.abs(check-original).max()))
        for local,(path,predicted_cost) in enumerate(zip(xy,original)):
            i=start+local
            ident=f'r{index+1:02d}_c{i:03d}'
            command=actions[i].numpy()
            first_xy=actual_xy[0]+2*command[0,:config.micro_action_dim]
            free_first_hit=(35<first_xy[0]<60 and 4<first_xy[1]<60
                            and np.linalg.norm(first_xy-goal)<SUCCESS_RADIUS)
            predicted_rows.append(predicted_row(ident,path,predicted_cost,round=index+1,
                                               elite=i in elite,free_first_hit=bool(free_first_hit)))
            commands_by_id[ident]=command
    print(f'Predicted paths for round {index+1}/{len(rounds)}')
assert score_error<=1e-5, f'Reconstructed sampled scores differ by {score_error}.'
print('Max sampled score reconstruction error:',score_error)

sampled=pd.DataFrame(predicted_rows)
sampled['predicted_delta_vs_returned']=sampled['predicted_mse']-plan.cost
sampled.to_csv('/content/cem_detour_sampled_candidates.csv',index=False)
np.savez_compressed('/content/cem_detour_sampled_sequences.npz',
                    actions=np.concatenate(command_blocks),predicted_xy=np.asarray(state_paths),
                    ids=sampled.id.to_numpy(dtype=str))
direct=sampled[sampled.low_detour].sort_values('predicted_mse')
print('Low-detour predicted successes:',len(direct),'of',len(sampled))
print('Low-detour candidates scoring below the returned sequence:',
      int((direct.predicted_delta_vs_returned < -1e-7).sum()))
print('Candidates whose first command would enter the circle in free motion:',int(sampled.free_first_hit.sum()))
display(direct.head(12).round(5))

## Replay a small set from the same reset

We replay the lowest-cost low-detour candidates, the earliest predicted arrival in that group, and the best sampled candidate in the final round. We also include up to two candidates whose first command would enter the circle under free motion, then verify those in the simulator. When the low-detour group is empty, we use the smallest-excursion predicted successes, or the smallest excursions overall if the model predicts no successes. The scripted control aims straight at the goal and pads with zero actions. It is scored separately from the CEM samples.

In [ ]:
selected={}

def include(ids,reason):
    for ident in ids:
        selected.setdefault(ident,[]).append(reason)

final=sampled[sampled['round']==len(rounds)]
include([final.loc[final.predicted_mse.idxmin(),'id']],'best final-round sample')
include([sampled.loc[sampled.predicted_mse.idxmin(),'id']],'best sample across rounds')
include(sampled[sampled.free_first_hit].nsmallest(2,'predicted_mse').id,'direct first command')
if len(direct):
    include(direct.head(3).id,'low detour, low cost')
    include(direct.sort_values(['predicted_first_hit','predicted_mse']).head(1).id,'earliest predicted low-detour arrival')
else:
    predicted_successes=sampled[sampled.predicted_first_hit.notna()]
    pool=predicted_successes if len(predicted_successes) else sampled
    include(pool.sort_values(['predicted_outward','predicted_extra','predicted_mse']).head(3).id,
            'smallest predicted excursion')

control=np.zeros((horizon*stride,config.micro_action_dim),dtype=np.float32)
# The pinned Wall simulator moves the ball center by twice the primitive action in free space.
straight=(goal-actual_xy[0])/2
needed=max(1,int(np.ceil(np.linalg.norm(straight)/planning['max_action_length'])))
assert needed<=len(control)
control[:needed]=straight/needed
control=control.reshape(horizon,-1)
special={'returned elite mean':macro,'scripted direct control':control}
rows_by_id={r['id']:r for r in predicted_rows}
for ident,actions in special.items():
    xy,cost=predict(torch.from_numpy(actions)[None])
    rows_by_id[ident]=predicted_row(ident,xy[0],plan.cost if ident=='returned elite mean' else cost[0],
                                   round=None,elite=False)

def replay(actions):
    initial=reset_case()
    positions=[initial.state[:2].copy()]
    frames=[initial.frame.copy()]
    first=None
    for i,action in enumerate(actions.reshape(-1,config.micro_action_dim),1):
        step=env.step(action)
        positions.append(step.state[:2].copy())
        frames.append(step.frame.copy())
        if step.success and first is None:
            first=i
    xy=np.asarray(positions)
    terminal=visual.encode_frame(encoder,frames[-1],device,precision)
    return xy,frames,first,float(score(terminal[None])[0])

results=[]
replays={}
for ident in [*special,*selected]:
    actions=special[ident] if ident in special else commands_by_id[ident]
    xy,frames,hit,mse=replay(actions)
    if ident=='returned elite mean':
        assert np.array_equal(xy[:steps+1],actual_xy)
        assert np.array_equal(np.stack(frames[:steps+1]),raw['frames'].numpy())
        assert hit==steps
    replays[ident]=(xy,hit)
    geom=geometry(xy,hit)
    results.append(dict(**rows_by_id[ident],why='; '.join(selected.get(ident,[])),
                        predicted_delta_vs_returned=rows_by_id[ident]['predicted_mse']-plan.cost,
                        actual_first_hit=hit,actual_path_to_hit_or_end=geom['path_length'],
                        actual_extra_to_hit_or_end=geom['extra'],actual_outward=geom['outward'],
                        actual_terminal_distance=float(np.linalg.norm(xy[-1]-goal)),actual_terminal_mse=mse))
table=pd.DataFrame(results)
table.to_csv('/content/cem_detour_replayed_candidates.csv',index=False)
display(table[['id','why','predicted_first_hit','predicted_outward','predicted_mse','predicted_delta_vs_returned',
               'actual_first_hit','actual_path_to_hit_or_end','actual_extra_to_hit_or_end',
               'actual_terminal_distance','actual_terminal_mse']].round(5))

## Inspect the tradeoff and real paths

The scatter shows every sampled plan and highlights the replayed subset. In the trajectory panels the solid segment ends at first success. The dotted segment continues to the endpoint CEM was scoring. For runs without success the solid line shows all executed actions.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

fig,ax=plt.subplots(figsize=(9,6),constrained_layout=True)
ax.scatter(sampled.predicted_outward,sampled.predicted_mse,s=9,c='gray',alpha=.25,label='All sampled')
elites=sampled[sampled.elite & (sampled['round']==len(rounds))]
ax.scatter(elites.predicted_outward,elites.predicted_mse,s=20,c='tab:blue',label='Final elites')
for ident,color,marker in [('returned elite mean','tab:red','*'),('scripted direct control','tab:green','D')]:
    row=rows_by_id[ident]
    ax.scatter(row['predicted_outward'],row['predicted_mse'],color=color,marker=marker,s=125,label=ident,zorder=4)
replayed=table[table.id.isin(selected)]
ax.scatter(replayed.predicted_outward,replayed.predicted_mse,s=65,facecolors='none',
           edgecolors='black',label='Sampled and replayed')
ax.set(xlabel='Predicted outward excursion before first success (units)',ylabel='Predicted terminal latent MSE')
ax.legend()
fig.savefig('/content/cem_detour_candidate_costs.png',dpi=150)
plt.close(fig)
display(Image(filename='/content/cem_detour_candidate_costs.png'))

cols=3
nrows=(len(replays)+cols-1)//cols
fig,axes=plt.subplots(nrows,cols,figsize=(13,4*nrows),squeeze=False,constrained_layout=True)
points=np.vstack([*(xy for xy,_ in replays.values()),goal-SUCCESS_RADIUS,goal+SUCCESS_RADIUS])
lo,hi=points.min(axis=0)-1,points.max(axis=0)+1
for ax,(ident,(xy,hit)) in zip(axes.flat,replays.items()):
    stop=len(xy)-1 if hit is None else hit
    ax.add_patch(Circle(goal,SUCCESS_RADIUS,fill=False,edgecolor='gray',ls=':'))
    ax.scatter(*goal,c='black',marker='*',s=100)
    ax.scatter(*xy[0],facecolors='none',edgecolors='tab:blue',s=50)
    ax.plot(*xy[:stop+1].T,'o-',color='tab:blue',ms=3)
    if hit is not None:
        ax.plot(*xy[hit:].T,'o:',color='tab:green',ms=3)
    ax.scatter(*xy[-1],marker='x',c='black',s=35)
    status='no success' if hit is None else f'first hit at {hit}'
    ax.set(xlim=(lo[0],hi[0]),ylim=(hi[1],lo[1]),aspect='equal',
           xlabel='Ball x',ylabel='Ball y',title=f'{ident} · {status}')
for ax in axes.flat[len(replays):]:
    ax.set_visible(False)
fig.savefig('/content/cem_detour_replayed_paths.png',dpi=150)
plt.close(fig)
display(Image(filename='/content/cem_detour_replayed_paths.png'))